# IDC Manifest And Patient Split Runner

This notebook keeps the implementation in `src/idc_pipeline/` and uses Colab only as the runtime layer.

Workflow:
- mount Drive for the project code, cached dataset archive, and saved artifacts
- download the dataset zip into Google Drive only if it is missing
- extract the zip into `/content`
- validate patch readability and size
- build the cleaned manifest and patient-level split pipeline


In [1]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')

# Drive keeps the project code and the saved CSV artifacts between Colab sessions.
PROJECT_ROOT = Path('/content/drive/MyDrive/Workshop UDL')
SRC_DIR = PROJECT_ROOT / 'src'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts'
DATASET_CACHE_DIR = PROJECT_ROOT / 'datasets'
DATASET_ARCHIVE_PATH = DATASET_CACHE_DIR / 'Breast Cancer Histopathology Dataset.zip'

# The extracted dataset lives in /content because runtime storage is faster than Drive for many small files.
RUNTIME_DIR = Path('/content/idc_runtime')
RUNTIME_EXTRACT_DIR = RUNTIME_DIR / 'dataset'
DATASET_URL = 'https://seafile.unistra.fr/d/6b88fcf6a556438d8f45/files/?p=%2FBreast%20Cancer%20Histopathology%20Dataset.zip&dl=1'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if not SRC_DIR.exists():
    raise FileNotFoundError(
        f'Source directory not found: {SRC_DIR}. '
        'Put the project folder in Google Drive so Colab can import src/idc_pipeline.'
    )

print(f'Project root: {PROJECT_ROOT}')
print(f'Source dir: {SRC_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Dataset archive cache: {DATASET_ARCHIVE_PATH}')
print(f'Runtime extract dir: {RUNTIME_EXTRACT_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/Workshop UDL
Source dir: /content/drive/MyDrive/Workshop UDL/src
Output dir: /content/drive/MyDrive/Workshop UDL/artifacts
Dataset archive cache: /content/drive/MyDrive/Workshop UDL/datasets/Breast Cancer Histopathology Dataset.zip
Runtime extract dir: /content/idc_runtime/dataset


In [2]:
from idc_pipeline import SplitConfig, prepare_dataset_from_zip_url_with_cache, run_manifest_pipeline

dataset_dir = prepare_dataset_from_zip_url_with_cache(
    url=DATASET_URL,
    archive_path=DATASET_ARCHIVE_PATH,
    extract_dir=RUNTIME_EXTRACT_DIR,
)

print(f'Dataset ready at: {dataset_dir}')


Using existing archive: /content/drive/MyDrive/Workshop UDL/datasets/Breast Cancer Histopathology Dataset.zip
Using existing extracted dataset: /content/idc_runtime/dataset/IDC_regular_ps50_idx5
Resolved canonical dataset directory: /content/idc_runtime/dataset/IDC_regular_ps50_idx5
Dataset ready at: /content/idc_runtime/dataset/IDC_regular_ps50_idx5


In [3]:
# The split is patient-level, not patch-level, to avoid leakage between train/val/test.
split_config = SplitConfig(
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
    trials=40,
)

result = run_manifest_pipeline(
    dataset_dir=dataset_dir,
    output_dir=OUTPUT_DIR,
    split_config=split_config,
)

result


PipelineResult(dataset_dir=PosixPath('/content/idc_runtime/dataset/IDC_regular_ps50_idx5'), output_dir=PosixPath('/content/drive/MyDrive/Workshop UDL/artifacts'), manifest_path=PosixPath('/content/drive/MyDrive/Workshop UDL/artifacts/manifest.csv'), patient_summary_path=PosixPath('/content/drive/MyDrive/Workshop UDL/artifacts/patient_summary.csv'), manifest_with_split_path=PosixPath('/content/drive/MyDrive/Workshop UDL/artifacts/manifest_with_split.csv'), split_summary_path=PosixPath('/content/drive/MyDrive/Workshop UDL/artifacts/split_summary.csv'), cleaning_summary_path=PosixPath('/content/drive/MyDrive/Workshop UDL/artifacts/cleaning_summary.json'), invalid_samples_path=PosixPath('/content/drive/MyDrive/Workshop UDL/artifacts/invalid_samples.csv'), split_paths={'train': PosixPath('/content/drive/MyDrive/Workshop UDL/artifacts/splits/train.csv'), 'val': PosixPath('/content/drive/MyDrive/Workshop UDL/artifacts/splits/val.csv'), 'test': PosixPath('/content/drive/MyDrive/Workshop UDL/ar

In [4]:
import json

print(json.dumps(result.to_dict(), indent=2))


{
  "dataset_dir": "/content/idc_runtime/dataset/IDC_regular_ps50_idx5",
  "output_dir": "/content/drive/MyDrive/Workshop UDL/artifacts",
  "expected_patch_size": {
    "width": 50,
    "height": 50
  },
  "scanned_patient_count": 279,
  "patient_count": 279,
  "scanned_image_count": 277524,
  "image_count": 275222,
  "dropped_image_count": 2302,
  "dropped_reason_counts": {
    "wrong_size": 2302
  },
  "split_score": 0.000411,
  "artifacts": {
    "manifest": "/content/drive/MyDrive/Workshop UDL/artifacts/manifest.csv",
    "patient_summary": "/content/drive/MyDrive/Workshop UDL/artifacts/patient_summary.csv",
    "manifest_with_split": "/content/drive/MyDrive/Workshop UDL/artifacts/manifest_with_split.csv",
    "split_summary": "/content/drive/MyDrive/Workshop UDL/artifacts/split_summary.csv",
    "cleaning_summary": "/content/drive/MyDrive/Workshop UDL/artifacts/cleaning_summary.json",
    "invalid_samples": "/content/drive/MyDrive/Workshop UDL/artifacts/invalid_samples.csv",
    "

## Outputs

Saved to `PROJECT_ROOT / 'artifacts'`:

- `cleaning_summary.json`
- `invalid_samples.csv`
- `manifest.csv`
- `patient_summary.csv`
- `manifest_with_split.csv`
- `split_summary.csv`
- `splits/train.csv`, `splits/val.csv`, `splits/test.csv`

Dataset storage strategy:
- cached zip in `PROJECT_ROOT / 'datasets'`
- extracted image tree in `/content/idc_runtime`

The raw dataset is left untouched. Invalid patches are excluded at the manifest level and recorded in the cleaning artifacts.
